In [ ]:
"""
Robot Controller - Integrates with ObjectDetector for autonomous picking.
Uses real-world coordinates from the camera for precise arm positioning.
"""

import time
import numpy as np
from Arm_Lib import Arm_Device
import sys
import os

# Add parent folder to path to import coord_converter
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from coord_converter import pixel_to_shelf

# ============= ROBOT CONFIGURATION =============
# DofBot workspace limits (in mm, relative to robot base)
ROBOT_X_LIMITS = (50, 300)       # Forward/backward
ROBOT_Y_LIMITS = (-200, 200)     # Left/right
ROBOT_Z_LIMITS = (50, 350)       # Up/down

# Approach and grasp offsets (mm)
APPROACH_HEIGHT_OFFSET = 80      # How high above object to approach
GRASP_HEIGHT_OFFSET = 10         # How low to go for grasping

# Home/safe positions
HOME_POSITION = (150, 0, 250)
DROP_ZONE = (100, -150, 150)    # Where to drop picked objects

# Servo angle limits
SERVO_ANGLES_MIN = [0, 40, 10, 0, 0, 0]      # Min angles for each servo
SERVO_ANGLES_MAX = [180, 170, 170, 180, 180, 180]  # Max angles for each servo
MOVE_TIME_MS = 800               # Default movement time in ms

In [ ]:
class RobotController:
    """
    Controls the DofBot robot arm with coordinate conversion and safety checks.
    
    Accepts real-world coordinates from object detection and converts them
    to robot servo angles for precise picking operations.
    """

    def __init__(self, use_real_hardware=True):
        """
        Initialize robot controller.
        
        Args:
            use_real_hardware: If True, initialize actual Arm_Device. 
                             If False, simulate commands (for testing).
        """
        print("🤖 Initializing robot controller...")
        self.is_connected = False
        self.arm = None
        self.use_real_hardware = use_real_hardware
        
        if use_real_hardware:
            try:
                self.arm = Arm_Device()
                time.sleep(0.2)
                print("✓ Arm_Device initialized")
            except Exception as e:
                print(f"⚠ Could not initialize Arm_Device: {e}")
                print("  Running in simulation mode")
                self.use_real_hardware = False
    
    def beep(self, duration_ms=500):
        """
        Sound a buzzer beep.
        
        Args:
            duration_ms: Duration of beep in milliseconds
        """
        if self.use_real_hardware and self.arm is not None:
            try:
                self.arm.Arm_Buzzer_On(duration_ms)
                time.sleep(duration_ms / 1000.0)
                self.arm.Arm_Buzzer_On(0)  # Turn off buzzer
            except Exception as e:
                print(f"⚠ Buzzer control failed: {e}")
        else:
            print(f"  [SIM] BEEP! (duration: {duration_ms}ms)")
        
    def connect(self):
        """Connect to robot and initialize home position."""
        if self.use_real_hardware and self.arm is None:
            print("❌ Cannot connect - Arm_Device not initialized")
            return False
        
        print("✓ Robot connected")
        self.beep()  # Beep on connection
        self.is_connected = True
        return True

    def disconnect(self):
        """Safely disconnect from robot."""
        self.beep()  # Beep before disconnect
        if self.arm is not None:
            self.arm = None
        print("Robot disconnected")
        self.is_connected = False

    def _execute_move(self, angles, move_time=MOVE_TIME_MS):
        """
        Execute servo movement with angle validation.
        
        Args:
            angles: List of 6 servo angles [base, shoulder, elbow, wrist, wrist_rotate, gripper]
            move_time: Movement duration in milliseconds
            
        Returns:
            bool: True if successful
        """
        # Validate angles
        for i, angle in enumerate(angles):
            if not (SERVO_ANGLES_MIN[i] <= angle <= SERVO_ANGLES_MAX[i]):
                print(f"⚠ Servo {i} angle {angle}° outside limits [{SERVO_ANGLES_MIN[i]}, {SERVO_ANGLES_MAX[i]}]")
                return False
        
        # Beep before move
        self.beep(200)
        
        if self.use_real_hardware and self.arm is not None:
            try:
                self.arm.Arm_serial_servo_write6(*angles, move_time)
                time.sleep(move_time / 1000.0)  # Wait for motion to complete
                # Beep after move
                self.beep(200)
                return True
            except Exception as e:
                print(f"❌ Movement failed: {e}")
                return False
        else:
            # Simulation mode - just print the command
            print(f"  [SIM] Move: angles={angles}, time={move_time}ms")
            time.sleep(0.5)
            # Beep after move
            self.beep(200)
            return True

    def move_to(self, x_mm, y_mm, z_mm, gripper_open=True, move_time=MOVE_TIME_MS):
        """
        Move robot to position in real-world coordinates.
        
        Args:
            x_mm, y_mm, z_mm: Target position in mm (robot base frame)
            gripper_open: True for open, False for closed
            move_time: Movement duration in ms
            
        Returns:
            bool: Success
        """
        # Safety checks
        if not (ROBOT_X_LIMITS[0] <= x_mm <= ROBOT_X_LIMITS[1]):
            print(f"⚠ X={x_mm:.1f}mm outside limits {ROBOT_X_LIMITS}")
            return False
        if not (ROBOT_Y_LIMITS[0] <= y_mm <= ROBOT_Y_LIMITS[1]):
            print(f"⚠ Y={y_mm:.1f}mm outside limits {ROBOT_Y_LIMITS}")
            return False
        if not (ROBOT_Z_LIMITS[0] <= z_mm <= ROBOT_Z_LIMITS[1]):
            print(f"⚠ Z={z_mm:.1f}mm outside limits {ROBOT_Z_LIMITS}")
            return False

        print(f"📍 Moving to X:{x_mm:.0f}mm Y:{y_mm:.0f}mm Z:{z_mm:.0f}mm | Gripper:{'OPEN' if gripper_open else 'CLOSED'}")

        # TODO: Implement inverse kinematics to convert (x, y, z) → servo angles
        # For now, using simulated angles
        # Example implementation:
        # angles = self.inverse_kinematics(x_mm, y_mm, z_mm)
        
        # Placeholder angles for testing
        angles = [90, 90, 90, 0, 90, 0 if gripper_open else 50]
        
        return self._execute_move(angles, move_time)

    def set_gripper(self, open_state, move_time=500):
        """
        Open or close gripper.
        
        Args:
            open_state: True for open, False for closed
            move_time: Movement duration in ms
        """
        gripper_angle = 0 if open_state else 50
        print(f"🔧 Gripper: {'OPEN' if open_state else 'CLOSE'}")
        
        # Servo 6 is the gripper
        if self.use_real_hardware and self.arm is not None:
            try:
                self.arm.Arm_serial_servo_write(6, gripper_angle, move_time)
                time.sleep(move_time / 1000.0)
            except Exception as e:
                print(f"❌ Gripper control failed: {e}")
        else:
            print(f"  [SIM] Gripper angle={gripper_angle}°")
            time.sleep(move_time / 1000.0)

    def home(self, move_time=1000):
        """
        Move to home position (safe, neutral pose).
        
        Args:
            move_time: Movement duration in ms
        """
        print("↩️  Moving to HOME position...")
        x, y, z = HOME_POSITION
        return self.move_to(x, y, z, gripper_open=True, move_time=move_time)

    def pick_object(self, x_mm, y_mm, z_mm, obj_label="object"):
        """
        Execute complete autonomous pick sequence from detected object.
        
        Args:
            x_mm, y_mm, z_mm: Object position in real-world coordinates (mm)
            obj_label: Label of detected object (for logging)
            
        Returns:
            bool: True if pick successful
        """
        print(f"\n{'='*60}")
        print(f"🤖 PICK SEQUENCE: {obj_label.upper()}")
        print(f"   Target: X={x_mm:.1f}mm Y={y_mm:.1f}mm Z={z_mm:.1f}mm")
        print(f"{'='*60}")

        # 1. Approach above object

        approach_z = z_mm + APPROACH_HEIGHT_OFFSET
        print(f"[1/5] Approaching above object...")
        if not self.move_to(x_mm, y_mm, approach_z, gripper_open=True):
            print("❌ Approach failed")
            return False
        print("     ✓ Approached")

        # 2. Lower to grasp height
        grasp_z = z_mm + GRASP_HEIGHT_OFFSET
        print(f"[2/5] Lowering to grasp height...")
        if not self.move_to(x_mm, y_mm, grasp_z, gripper_open=True):
            print("❌ Lower failed")
            return False
        print("     ✓ Positioned")

        # 3. Close gripper
        print(f"[3/5] Closing gripper...")
        self.set_gripper(False)
        print("     ✓ Gripped")

        # 4. Lift object
        print(f"[4/5] Lifting object...")
        self.move_to(x_mm, y_mm, approach_z, gripper_open=False)
        print("     ✓ Lifted")

        # 5. Move to drop zone
        print(f"[5/5] Moving to drop zone...")
        drop_x, drop_y, drop_z = DROP_ZONE
        self.move_to(drop_x, drop_y, drop_z, gripper_open=False)
        print("     ✓ At drop zone")

        # Release object
        print(f"[6/5] Releasing object...")
        self.set_gripper(True)
        print("     ✓ Released")

        # Return to home
        print(f"[7/5] Returning to home...")
        self.home()
        
        print(f"{'='*60}")
        print(f"✅ PICK COMPLETE - {obj_label.upper()} picked successfully!")
        print(f"{'='*60}\n")

        return True

    def pick_detected_object(self, detection_data):
        """
        Pick an object using detection data from ObjectDetector.
        
        Args:
            detection_data: Dict with keys:
                - 'label': object class name
                - 'x': real-world X coordinate (mm)
                - 'y': real-world Y coordinate (mm)
                - 'z': real-world Z coordinate (mm)
                - 'confidence': detection confidence (optional)
        """
        label = detection_data.get('label', 'unknown')
        x = detection_data.get('x')
        y = detection_data.get('y')
        z = detection_data.get('z')
        conf = detection_data.get('confidence', 0.0)

        if x is None or y is None or z is None:
            print("❌ Invalid detection data - missing coordinates")
            return False

        print(f"📦 Detected: {label} (confidence: {conf:.2f})")
        return self.pick_object(x, y, z, obj_label=label)

In [ ]:
# ============= INTEGRATED EXAMPLE: Detect and Pick =============
"""
This example shows how to:
1. Initialize the ObjectDetector
2. Capture and detect objects
3. Convert pixel coordinates to real-world coordinates
4. Command the robot to pick detected objects
"""

if __name__ == "__main__":
    # Import object detector from core module
    import sys
    sys.path.append(".")
    from object_detector import ObjectDetector3D
    
    # Initialize detector and robot
    print("="*60)
    print("AUTONOMOUS PICKING SYSTEM")
    print("="*60)
    
    detector = ObjectDetector3D(
        model_path="model/runs/train/toy_animals_full/weights/best.pt",
        camera_calibration_path="./calibration_files"
    )
    
    robot = RobotController(use_real_hardware=False)  # Set to True when using real hardware
    
    # Connect robot
    if not robot.connect():
        print("Failed to connect robot")
        exit(1)
    
    print("\n📹 Starting object detection...")
    print("Press SPACE to pick the detected object, ESC to exit\n")
    
    try:
        robot.home()  # Start from home position
        
        # Run live detection
        detector.open_camera()
        
        frame_count = 0
        detections = []
        
        while True:
            ret, frame = detector.cap.read()
            if not ret:
                continue
            
            frame_count += 1
            
            # Apply undistortion if available
            if detector.map1 is not None and detector.map2 is not None:
                frame = detector.cap.read(detector.map1, detector.map2, cv2.INTER_LINEAR)
            
            display = frame.copy()
            detector.draw_virtual_grid(display)
            
            # Run detection every N frames
            if frame_count % detector.INFERENCE_INTERVAL == 0:
                detections = detector.detect_animals(frame)
            
            # Draw detections
            for label, cx, cy, x1, y1, x2, y2, conf in detections:
                cv2.rectangle(display, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 2)
                cv2.circle(display, (cx, cy), 5, (0, 255, 0), -1)
                cv2.putText(display, f"{label} {conf:.2f}", (int(x1), int(y1) - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
            
            cv2.putText(display, f"Objects: {len(detections)} | Press SPACE to pick first | ESC to exit",
                        (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
            
            cv2.imshow("Detection + Robot Control", display)
            
            key = cv2.waitKey(1) & 0xFF
            if key == 27:  # ESC
                break
            elif key == 32 and detections:  # SPACE
                # Get first detected object
                label, cx, cy, x1, y1, x2, y2, conf = detections[0]
                bbox_height_pixels = y2 - y1
                
                # Convert to real-world coordinates
                X, Y, Z = detector.get_real_world_coords(cx, cy, bbox_height_pixels, label)
                
                if X is not None:
                    print(f"\n✓ Coordinates: X={X:.2f}mm, Y={Y:.2f}mm, Z={Z:.2f}mm")
                    
                    # Command robot to pick
                    detection_data = {
                        'label': label,
                        'x': X,
                        'y': Y,
                        'z': Z,
                        'confidence': conf
                    }
                    robot.pick_detected_object(detection_data)
                else:
                    print("❌ Could not compute coordinates")
                
                # Return to home for next pick
                time.sleep(1)
                robot.home()
        
        robot.disconnect()
        detector.close_camera()
        
    except KeyboardInterrupt:
        print("\n⛔ Interrupted by user")
        robot.disconnect()
        detector.close_camera()

Arm library loaded successfully
Open-box pose reached!


In [14]:
del Arm

In [11]:
from Arm_Lib import Arm_Device
import time

# Initialize arm
Arm = Arm_Device()
time.sleep(0.1)

# Open box left pose
Arm.Arm_serial_servo_write6(60, 120, 70, 70, 90, 50, 800)
time.sleep(0.8)  # wait for motion

